# 03 — Signal sensitivity per plane

For each candidate plane (survivors of gates 1–2, even half): signal efficiency into
A, xsec-independent sideband leakage (S_B/S_A etc.), the bias the leakage induces in
the background prediction as a function of signal strength, and the Asimov Z at the
1 fb reference with the notebook-02 non-closure systematic folded in. Signal
normalization uses the census gen-filter denominators (absolute ε is honest).

In [ ]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

sys.path.insert(0, os.getcwd())
import abcd_tools as at
import study_setup as ss

hep.style.use("CMS")
plt.rcParams["figure.figsize"] = (7, 5)
LUMI_LABEL = r"59.8 fb$^{-1}$ (13 TeV, 2018 sim.)"

def cms_label(ax=None):
    hep.cms.label("Work in progress", data=False, rlabel=LUMI_LABEL, ax=ax)

# pre-skim / pre-filter sums of gen weights (see README "Normalization")
SUMW_PRE = {}
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_2MU2E))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_4MU))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_BKG_UNSKIMMED))

# load everything, normalized (TTJets kept at the campaign 471.7 pb; the NNLO
# alternative is an explicit, documented rescale -- see the xsec table below)
bkg = {s: ss.load_normalized(s, SUMW_PRE)[0] for s in ss.BACKGROUNDS}
sig = {s: ss.load_normalized(s, SUMW_PRE)[0] for s in ss.SIGNALS_2MU2E + ss.SIGNALS_4MU}
by_process = {p: ss.sum_process(bkg, [s for s, pp in ss.BACKGROUNDS.items() if pp == p])
              for p in ss.PROCESSES}
total_bkg = ss.sum_process(bkg, list(ss.BACKGROUNDS))
print(f"loaded {len(bkg)} background + {len(sig)} signal samples")

In [ ]:
gates = json.load(open(os.path.join(ss.WORKDIR, "gates_even.json")))
results = {}
for ch, signals in [("2mu2e", ss.SIGNALS_2MU2E), ("4mu", ss.SIGNALS_4MU)]:
    for pname, spec in ss.PLANES[ch].items():
        g2 = gates["gate2"][f"{ch}/{pname}"]
        if min(g2["n_eff"].values()) < 10:
            continue  # failed gate 2
        bvals, bvar, xe, ye = ss.plane_arrays(total_bkg, ch, pname, parity=0)
        breg = at.region_sums(bvals, bvar, xe, ye, spec["xspec"], spec["yspec"])
        bpred, bpred_var = at.abcd_prediction(breg)
        syst = max([v for v in (gates["gate3_nonclosure"].get(f"{ch}/{pname}") or {}).values()
                    if v is not None] + [0.0])
        sigma_b = np.sqrt(bpred_var + (syst * bpred) ** 2)
        for s in signals:
            svals, svar, sxe, sye = ss.plane_arrays(sig[s], ch, pname, parity=0)
            sreg = at.region_sums(svals, svar, sxe, sye, spec["xspec"], spec["yspec"])
            lr = at.leakage_ratios(sreg)
            z = at.asimov_z(sreg["A"][0], bpred, sigma_b)
            results[f"{ch}/{pname}/{s}"] = {"S_A": sreg["A"][0], "B_pred": bpred,
                "sigma_B": sigma_b, "Z": z, "leakage": lr}
json.dump(results, open(os.path.join(ss.WORKDIR, "sensitivity_even.json"), "w"),
          indent=1, default=float)
print(f"{len(results)} plane x signal evaluations")

In [ ]:
# ranking summary: median Z across signal points per plane, worst-case leakage
import collections
agg = collections.defaultdict(list)
for key, r in results.items():
    ch, pname, s = key.split("/", 2)
    agg[f"{ch}/{pname}"].append((r["Z"], max(v for v in r["leakage"].values() if np.isfinite(v))))
print(f"{'plane':24s} {'median Z':>9s} {'max Z':>7s} {'worst leakage':>14s}")
for k, v in sorted(agg.items(), key=lambda kv: -np.median([x[0] for x in kv[1]])):
    zs = [x[0] for x in v]; ls = [x[1] for x in v]
    print(f"{k:24s} {np.median(zs):9.3f} {max(zs):7.3f} {max(ls):14.3f}")

## Per-point detail plots

*(bar charts per plane after execution — kept lean until the numbers exist)*